In [1]:
from microtorch.model_maker import ModelMaker
from microtool.microtorch import MicrotorchTissueModel

microtorch_model = ModelMaker("SphereBallAstrosticks")
sphere, ball, astrosticks = microtorch_model.compartments

# Sphere's diffusivity is fixed inside the compartment, not a model parameter.
sphere.fixed_D = 2

# Make Astrosticks use a fixed D_par of 9.0.
astrosticks.fixed_D_par = 9.0
astrosticks.parameter_ranges = [[9.0, 9.0]]
microtorch_model.parameter_ranges[-1, :] = [9.0, 9.0]

model = MicrotorchTissueModel(
    microtorch_model,
    parameters={
        "radius": 6.0,  # Sphere radius
        "D": 2.0,        # Ball diffusivity
        "D_par": 9.0,    # Astrosticks parallel diffusivity
        "f_0": 0.60,      # Sphere
        "f_1": 0.35,      # Ball
        "f_2": 0.05,      # Astrosticks; derived from f_0 and f_1
    },
)

# Keep the Ball and Astrosticks diffusivities fixed during fitting.
for name in ("D", "D_par"):
    model[name].optimize = False
    model[name].fit_flag = False

print(model)
print("Sphere fixed diffusivity:", sphere.fixed_D)

No YAML configuration found for SphereBallAstrosticks model.
Falling back to parsing model name for compartments: ['Sphere', 'Ball', 'Astrosticks'].
Parameter ranges will be the default compartment values.
-----------
########### Making model:  SphereBallAstrosticks
########### Compartments: ['Sphere', 'Ball', 'Astrosticks']
########### Parameter names: [['radius'], ['D'], ['D_par']]
########### Parameter ranges: [[[0.001, 15]], [[0.001, 3]], [[0.5, 3.0]]]
-----------
Tissue model with 7 scalar parameters:
Tissue-parameter      Value    Scale  Optimize    Fit    Fit Bounds
------------------  -------  -------  ----------  -----  -------------
radius                 6           1  True        True   (0.001, 15.0)
D                      2           1  False       False  (0.001, 3.0)
D_par                  9           1  False       False  (9.0, 9.0)
f_0                    0.6         1  True        True   (0.0, 1.0)
f_1                    0.35        1  True        True   (0.0, 1.0)
f_2 

In [ ]:
import numpy as np
from microtool.acquisition_scheme import DiffusionAcquisitionScheme
from microtool.gradient_sampling import sample_uniform_half_sphere

n_directions = 6
n_shells = 5
block_size = n_directions + 1  # b0 + n_directions

shell_bvals = [1500, 1000, 1000, 1000, 2500]
shell_deltas = [0.0258, 0.0205,0.0205,0.0205, 0.0334]
shell_Deltas = [0.0434, 0.0431,0.0431, 0.0431, 0.0455]
initial_te = 0.09

# Build the measurements in shell blocks: b0, then six directions.
b_vals_list = []
pulse_durations_list = []
pulse_intervals_list = []
echo_times_list = []

for i in range(n_shells):
    b_vals_list.append(0.0)
    b_vals_list.extend([shell_bvals[i]] * n_directions)

    pulse_durations_list.extend([shell_deltas[i]] * block_size)
    pulse_intervals_list.extend([shell_Deltas[i]] * block_size)
    echo_times_list.extend([initial_te] * block_size)

b_vals = np.array(b_vals_list)
pulse_durations = np.array(pulse_durations_list)
pulse_intervals = np.array(pulse_intervals_list)
echo_times = np.array(echo_times_list)
directions = sample_uniform_half_sphere(len(b_vals))

initial_scheme = DiffusionAcquisitionScheme.from_bvals(
    b_values=b_vals,
    b_vectors=directions,
    pulse_durations=pulse_durations,
    pulse_intervals=pulse_intervals,
    echo_times=echo_times,
)

# One optimized TE, delta, and Delta per shell, shared across that shell's
# b0 and diffusion-weighted measurements.
for name in (
    "EchoTime",
    "DiffusionPulseDuration",
    "DiffusionPulseInterval",
):
    initial_scheme[name].set_repetition_period(block_size)

# Keep b0 magnitudes fixed and share one optimized magnitude per shell.
b0_mask = np.isclose(b_vals, 0.0)
magnitude = initial_scheme["DiffusionPulseMagnitude"]
magnitude.set_fixed_mask(b0_mask)
magnitude.set_repetition_period(n_directions)

# Check the masks and b0 values.
print("Timing free indices:",
      np.flatnonzero(initial_scheme["EchoTime"].optimize_mask))
print("Gradient-magnitude free indices:",
      np.flatnonzero(magnitude.optimize_mask))
print("b0 gradient magnitudes:", magnitude.values[b0_mask])

assert np.all(magnitude.values[b0_mask] == 0.0)
assert not np.any(magnitude.optimize_mask[b0_mask])
assert np.array_equal(
    np.flatnonzero(magnitude.optimize_mask),
    np.arange(n_shells) * block_size + 1,
)

print(initial_scheme)


In [3]:
signal = model(initial_scheme)
print(signal)

[1.         0.58170921 0.58170921 0.58170921 0.58170921 0.58170921
 0.58170921 1.         0.6251443  0.6251443  0.6251443  0.6251443
 0.6251443  0.6251443  1.         0.6251443  0.6251443  0.6251443
 0.6251443  0.6251443  0.6251443  1.         0.6251443  0.6251443
 0.6251443  0.6251443  0.6251443  0.6251443  1.         0.54781884
 0.54781884 0.54781884 0.54781884 0.54781884 0.54781884]


In [4]:
jacobian = model.scaled_jacobian(initial_scheme)
print(jacobian)
print(jacobian.shape)

[[ 0.          0.          0.        ]
 [-0.02754953  0.67917234 -0.19141331]
 [-0.02754953  0.67917234 -0.19141331]
 [-0.02754953  0.67917234 -0.19141331]
 [-0.02754953  0.67917234 -0.19141331]
 [-0.02754953  0.67917234 -0.19141331]
 [-0.02754953  0.67917234 -0.19141331]
 [ 0.          0.          0.        ]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [ 0.          0.          0.        ]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [ 0.          0.          0.        ]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219 -0.16006716]
 [-0.02094623  0.64294219

/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/torch/autograd/graph.py:1113: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [5]:
from microtool.optimize import optimize_scheme

optimal_scheme, _ = optimize_scheme(initial_scheme, model, noise_variance=0.05, method="trust-constr", solver_options = {
    "verbose": 2,
    "maxiter": 150,
    #"finite_diff_rel_step": 1e-8,  # Prevents large jumps during gradient estimation
    #"initial_tr_radius": 0.05        # Prevents trust radius from jumping to 35.0
})

| niter |f evals|CG iter|  obj func   |tr radius |   opt    |  c viol  |
|-------|-------|-------|-------------|----------|----------|----------|
|   1   |  46   |   0   | +9.0386e+01 | 1.00e+00 | 7.15e+01 | 0.00e+00 |


/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/scipy/optimize/_differentiable_functions.py:737: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)


|   2   |  92   |   1   | +5.4254e+01 | 1.00e+00 | 1.40e+02 | 0.00e+00 |
|   3   |  138  |  10   | +5.4254e+01 | 5.00e-01 | 1.40e+02 | 0.00e+00 |
|   4   |  184  |  19   | +5.4254e+01 | 2.50e-01 | 1.40e+02 | 0.00e+00 |
|   5   |  230  |  29   | +5.4254e+01 | 1.00e-01 | 1.40e+02 | 0.00e+00 |
|   6   |  276  |  35   | +5.9401e+01 | 1.00e-01 | 7.56e+01 | 0.00e+00 |
|   7   |  322  |  44   | +4.8415e+01 | 7.01e-01 | 4.48e+01 | 0.00e+00 |
|   8   |  368  |  57   | +5.2495e+01 | 4.91e+00 | 3.15e+02 | 0.00e+00 |
|   9   |  414  |  73   | +6.6392e+01 | 4.91e+00 | 1.04e+02 | 0.00e+00 |
|  10   |  460  |  91   | +4.3003e+01 | 8.87e+00 | 1.08e+02 | 0.00e+00 |
|  11   |  506  |  109  | +3.3671e+01 | 1.17e+01 | 1.81e+01 | 0.00e+00 |
|  12   |  552  |  131  | +2.9895e+01 | 1.17e+01 | 5.83e+01 | 0.00e+00 |
|  13   |  598  |  148  | +2.9167e+01 | 1.17e+01 | 6.32e+01 | 0.00e+00 |
|  14   |  644  |  165  | +2.8425e+01 | 1.17e+01 | 9.91e+01 | 0.00e+00 |
|  15   |  690  |  182  | +2.8053e+01 | 1.17e+01 | 

/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/scipy/optimize/_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)


|  149  | 9154  | 3482  | +1.9094e+01 | 1.84e-04 | 1.50e+02 | 0.00e+00 |
|  150  | 9246  | 3486  | +1.9094e+01 | 9.21e-05 | 1.50e+02 | 0.00e+00 |

The maximum number of function evaluations is exceeded.
Number of iterations: 150, function evaluations: 9246, CG iterations: 3486, optimality: 1.50e+02, constraint violation: 0.00e+00, execution time: 3.5e+02 s.
           message: The maximum number of function evaluations is exceeded.
           success: False
            status: 0
               fun: 19.094311412017575
                 x: [ 2.348e-02  2.290e-02 ...  9.106e+00  8.836e+00]
               nit: 150
              nfev: 9246
              njev: 201
              nhev: 0
          cg_niter: 3486
      cg_stop_cond: 2
              grad: [-8.631e+00  1.501e+00 ...  0.000e+00  0.000e+00]
   lagrangian_grad: [ 2.245e-02  2.274e-02 ... -1.867e+00 -1.498e+02]
            constr: [array([ 1.216e+00,  1.216e+00, ...,  6.053e-01,
                            6.053e-01]), array([ 4.000e-

/smb/user/jgroot34/DS-Data/Radiotherapie/Research/User/jgroot34/D_Projects/MICROtool/MICROtool_Github/microtool/microtool/optimize/optimize.py:246: UserWarning: Optimization procedure was unsuccessful. Possible solutions include but are not limited to: Changing the optimizer settings, changing the optimization method or changing the initial scheme to a more suitable one. If you are using a scipy optimizer its settings can be changed by passing options to this function. If you are using a MICROtool optimization method please consult the optimization_methods module for more details.
  warnings.warn(


In [7]:
print(optimal_scheme)
print(optimal_scheme.b_values)

Acquisition scheme with 35 measurements and 6 scalar parameters. 
total number of optimized parameters is 45:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.065)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.001, 0.1)    DiffusionPulseInterval [s] in (0.001, 0.1)    EchoTime [s] in (0.001, 0.1)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  --------------------------------------------  --------------------------------------------  ------------------------------
                                       -0                                          1.92731                                     1.5708                                         0.0254382                                     0.0439085                       0.0899801
                                        0.0234838                                  2.2988               